In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import sys
sys.path.append("../../utils/")

from utils import *


/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = 'CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1'
CARPETA_DATASET = 'CIC18__split__v1'

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_FOLDS_CSV = f"{NOMBRE_EXPERIMENTO}__folds_B.csv"
NOMBRE_RESULTADOS_HOLDOUT_CSV = f"{NOMBRE_EXPERIMENTO}__holdout_C.csv"
NOMBRE_RESULTADOS_RESUMEN_CSV = f"{NOMBRE_EXPERIMENTO}__tabla_resumen_B_y_C.csv"
NOMBRE_RESULTADOS_RESUMEN_JSON = f"{NOMBRE_EXPERIMENTO}__summary_B_y_C.json"

# ===== PARAMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== FASE 1 =====
N_VALUES = [10000]

# Cambia aqui el escalador del experimento:
# "StandardScaler" o "RobustScaler"
SCALER_NAME = 'RobustScaler'

# Cambia aqui la tecnica del experimento:
# "NONE", "RUS_SMOTE", "RUS_SMOTE_ENN", "NearMiss_SMOTE", "NearMiss_SMOTE_ENN"
ESTRATEGIA_DE_REBALANCEO = 'RUS_SMOTE'

DESCRIPCION_EXPERIMENTO = f"{SCALER_NAME} + {ESTRATEGIA_DE_REBALANCEO.replace('_', '/')}"

# ===== CONFIG MLP =====
MLP_HIDDEN_LAYER_SIZES = (100,)
MLP_ACTIVATION = "relu"
MLP_SOLVER = "adam"
MLP_ALPHA = 0.0001
MLP_BATCH_SIZE = "auto"
MLP_LEARNING_RATE = "adaptive"
MLP_LEARNING_RATE_INIT = 0.001
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_VALIDATION_FRACTION = 0.1
MLP_N_ITER_NO_CHANGE = 10
MLP_VERBOSE = False
MODEL_STEP_NAME = "mlp"
MODEL_DISPLAY_NAME = "MLPClassifier"
MODEL_PARAMS = {
    "hidden_layer_sizes": list(MLP_HIDDEN_LAYER_SIZES),
    "activation": MLP_ACTIVATION,
    "solver": MLP_SOLVER,
    "alpha": MLP_ALPHA,
    "batch_size": MLP_BATCH_SIZE,
    "learning_rate": MLP_LEARNING_RATE,
    "learning_rate_init": MLP_LEARNING_RATE_INIT,
    "max_iter": MLP_MAX_ITER,
    "early_stopping": MLP_EARLY_STOPPING,
    "validation_fraction": MLP_VALIDATION_FRACTION,
    "n_iter_no_change": MLP_N_ITER_NO_CHANGE,
    "verbose": MLP_VERBOSE
}

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 3

# ===== CONFIG REBALANCEO =====
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3


In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_FASE_1/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_FASE_1/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(1341149, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,2,0,44751,3,13,6733,6000,1100,0,41677,...,250,3,0,0,0,0,0,0,0,1
1,37274,4,753825,754,1064,6266,18066,1424,184,20085,...,4725,278,72650,56255,70259,43755,32542,11323,36885,3
2,2,0,4380198,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,7
3,624,0,9183,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,0
4,2,0,66089,3,13,379,1459,225,0,488,...,156,3,0,0,0,0,0,0,0,2


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
1,360000
0,360000
2,159089
3,116159
4,115628
5,111820
6,75238
7,33125
8,7926


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (1341149, 54)
Shape y_train: (1341149,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

print()
print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Forma del dataset test:
(335288, 55)
Columnas no numéricas en X_test:
[]

Distribución de clases en test:


,count
LABEL,
0,90000
1,90000
2,39772
3,29040
4,28907
5,27955
6,18810
7,8281
8,1982


In [9]:
def crear_scaler(nombre_scaler):
    if nombre_scaler == "StandardScaler":
        return StandardScaler()
    elif nombre_scaler == "RobustScaler":
        return RobustScaler()
    else:
        raise ValueError(f"Scaler no soportado: {nombre_scaler}")


def crear_modelo():
    return MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
        activation=MLP_ACTIVATION,
        solver=MLP_SOLVER,
        alpha=MLP_ALPHA,
        batch_size=MLP_BATCH_SIZE,
        learning_rate=MLP_LEARNING_RATE,
        learning_rate_init=MLP_LEARNING_RATE_INIT,
        max_iter=MLP_MAX_ITER,
        early_stopping=MLP_EARLY_STOPPING,
        validation_fraction=MLP_VALIDATION_FRACTION,
        n_iter_no_change=MLP_N_ITER_NO_CHANGE,
        verbose=MLP_VERBOSE,
        random_state=RANDOM_STATE
    )


def crear_pipeline():
    return Pipeline([
        ("scaler", crear_scaler(SCALER_NAME)),
        ("pca", PCA(n_components=N_COMPONENTS_PCA, random_state=RANDOM_STATE)),
        (MODEL_STEP_NAME, crear_modelo())
    ])


def calcular_fpr_fnr_multiclase(y_true, y_pred, labels_globales):
    cm = confusion_matrix(y_true, y_pred, labels=labels_globales)
    total = cm.sum()
    soporte = cm.sum(axis=1)

    fpr_por_clase = []
    fnr_por_clase = []

    for i in range(len(labels_globales)):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        fpr_por_clase.append(fpr)
        fnr_por_clase.append(fnr)

    fpr_por_clase = np.array(fpr_por_clase, dtype=float)
    fnr_por_clase = np.array(fnr_por_clase, dtype=float)
    soporte = np.array(soporte, dtype=float)

    if soporte.sum() > 0:
        fpr_weighted = np.average(fpr_por_clase, weights=soporte)
        fnr_weighted = np.average(fnr_por_clase, weights=soporte)
    else:
        fpr_weighted = 0.0
        fnr_weighted = 0.0

    return {
        "fpr_macro": float(np.mean(fpr_por_clase)),
        "fnr_macro": float(np.mean(fnr_por_clase)),
        "fpr_weighted": float(fpr_weighted),
        "fnr_weighted": float(fnr_weighted),
    }


def calcular_metricas_clasificacion(modelo, X_eval, y_eval, y_pred, labels_globales, score_time):
    metricas_fpr_fnr = calcular_fpr_fnr_multiclase(
        y_true=y_eval,
        y_pred=y_pred,
        labels_globales=labels_globales
    )

    metricas = {
        "accuracy": accuracy_score(y_eval, y_pred),

        "precision_weighted": precision_score(y_eval, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_eval, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_eval, y_pred, average="weighted", zero_division=0),

        "precision_macro": precision_score(y_eval, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_eval, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_eval, y_pred, average="macro", zero_division=0),

        "mcc": matthews_corrcoef(y_eval, y_pred),

        "roc_auc": calcular_roc_auc_multiclase_seguro(
            modelo=modelo,
            X_val=X_eval,
            y_val=y_eval,
            labels_globales=labels_globales,
            verbose=True
        ),

        "detection_latency_seconds": float(score_time / len(y_eval)) if len(y_eval) > 0 else 0.0,
    }

    metricas.update(metricas_fpr_fnr)
    return metricas


def resumir_metricas_cv(df, prefijo):
    """
    Resumen para validacion CV:
    devuelve media, desviacion tipica y varianza entre folds.
    """
    metricas = [
        "accuracy",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "fpr_weighted",
        "fnr_weighted",
        "fpr_macro",
        "fnr_macro",
        "mcc",
        "roc_auc",
        "detection_latency_seconds",
        "fit_time",
        "score_time"
    ]

    resumen = {}
    for metrica in metricas:
        col = f"{prefijo}_{metrica}"
        if col in df.columns:
            resumen[f"{metrica}_mean"] = float(df[col].mean())
            resumen[f"{metrica}_std"] = float(df[col].std(ddof=1))
            resumen[f"{metrica}_var"] = float(df[col].var(ddof=1))
    return resumen


def extraer_metricas_holdout(fila_holdout):
    """
    Resumen para hold-out final:
    devuelve un unico valor real, sin media ni desviacion tipica.
    """
    metricas = [
        "accuracy",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "fpr_weighted",
        "fnr_weighted",
        "fpr_macro",
        "fnr_macro",
        "mcc",
        "roc_auc",
        "detection_latency_seconds",
        "fit_time",
        "score_time"
    ]

    resumen = {}
    for metrica in metricas:
        col = f"C_{metrica}"
        if col in fila_holdout.index:
            resumen[metrica] = float(fila_holdout[col])
    return resumen


In [10]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [11]:
labels_globales = np.array(sorted(pd.unique(pd.concat([y_train, y_test]))))

resultados_folds = []
resultados_holdout = []

for target_n in N_VALUES:

    print("\n" + "#" * 100)
    print(f"N = {target_n} instancias por clase")
    print("#" * 100)

    # =====================================================================================
    # 1) CROSS-VALIDATION SOLO CON EL 80% DE TRAIN
    #    En cada fold:
    #    - A = 64% entrenamiento
    #    - B = 16% validación
    #    El hold-out C NO se toca aquí.
    # =====================================================================================
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

        print("=" * 80)
        print(f"N={target_n} | CV FOLD {fold}/{N_SPLITS}")
        print("=" * 80)

        # =========================
        # Split del fold: A y B
        # =========================
        df_train_fold = df_train.iloc[train_idx].copy()  # A
        df_val_fold = df_train.iloc[val_idx].copy()      # B

        # =========================
        # Rebalanceo SOLO sobre A
        # =========================
        df_train_fold_balanceado = rebalancear_train_fold(
            df_fold_train=df_train_fold,
            label_col=LABEL_COL,
            target_n=target_n,
            random_state=RANDOM_STATE + fold,
            nearmiss_version=NEARMISS_VERSION,
            smote_k_neighbors=SMOTE_K_NEIGHBORS,
            estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
            enn_n_neighbors=ENN_N_NEIGHBORS
        )

        X_A = df_train_fold_balanceado.drop(columns=[LABEL_COL])
        y_A = df_train_fold_balanceado[LABEL_COL]

        X_B = df_val_fold.drop(columns=[LABEL_COL])
        y_B = df_val_fold[LABEL_COL]

        # =========================
        # Modelo nuevo para cada fold
        # Pipeline:
        # - fit scaler con A
        # - fit PCA con A
        # - fit modelo con A
        # - evaluación solo en B
        # =========================
        pipeline_fold = crear_pipeline()

        t0 = time.time()
        pipeline_fold.fit(X_A, y_A)
        fit_time = time.time() - t0

        print("Clases en A:", sorted(pd.unique(y_A)))
        print("Clases en B:", sorted(pd.unique(y_B)))
        print("Clases del modelo:", pipeline_fold.named_steps[MODEL_STEP_NAME].classes_)
        print("Shape predict_proba B:", pipeline_fold.predict_proba(X_B).shape)

        # =========================
        # Validación sobre B
        # =========================
        t0 = time.time()
        y_pred_B = pipeline_fold.predict(X_B)
        score_time_B = time.time() - t0

        metricas_B = calcular_metricas_clasificacion(
            modelo=pipeline_fold,
            X_eval=X_B,
            y_eval=y_B,
            y_pred=y_pred_B,
            labels_globales=labels_globales,
            score_time=score_time_B
        )

        fila = {
            "numero_experimento": 1,
            "descripcion_experimento": DESCRIPCION_EXPERIMENTO,
            "scaler": SCALER_NAME,
            "rebalanceo": ESTRATEGIA_DE_REBALANCEO,
            "n": int(target_n),
            "fold": int(fold),

            "train_original_rows_A": int(df_train_fold.shape[0]),
            "train_balanceado_rows_A": int(df_train_fold_balanceado.shape[0]),
            "val_rows_B": int(df_val_fold.shape[0]),

            "B_fit_time": float(fit_time),
            "B_score_time": float(score_time_B),
        }

        for nombre, valor in metricas_B.items():
            fila[f"B_{nombre}"] = valor

        resultados_folds.append(fila)

        print(
            f"B -> acc={metricas_B['accuracy']:.6f}, f1_macro={metricas_B['f1_macro']:.6f}, "
            f"mcc={metricas_B['mcc']:.6f}, fpr_macro={metricas_B['fpr_macro']:.6f}, "
            f"fnr_macro={metricas_B['fnr_macro']:.6f}, auc={metricas_B['roc_auc']:.6f}, "
            f"latency={metricas_B['detection_latency_seconds']:.9f}s"
        )
        print()

    # =====================================================================================
    # 2) HOLD-OUT FINAL FUERA DEL CROSS-VALIDATION
    #    Una vez terminado el CV para este N:
    #    - se rebalancea TODO el 80% de entrenamiento
    #    - se entrena un modelo final con TODO ese 80%
    #    - se evalúa una única vez sobre el 20% de test C
    # =====================================================================================
    print("=" * 80)
    print(f"N={target_n} | MODELO FINAL SOBRE 80% TRAIN + HOLD-OUT 20% TEST")
    print("=" * 80)

    df_train_final_balanceado = rebalancear_train_fold(
        df_fold_train=df_train.copy(),
        label_col=LABEL_COL,
        target_n=target_n,
        random_state=RANDOM_STATE,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_final = df_train_final_balanceado.drop(columns=[LABEL_COL])
    y_train_final = df_train_final_balanceado[LABEL_COL]

    X_C = X_test
    y_C = y_test

    pipeline_final = crear_pipeline()

    t0 = time.time()
    pipeline_final.fit(X_train_final, y_train_final)
    fit_time_final = time.time() - t0

    print("Clases en train final:", sorted(pd.unique(y_train_final)))
    print("Clases en C:", sorted(pd.unique(y_C)))
    print("Clases del modelo final:", pipeline_final.named_steps[MODEL_STEP_NAME].classes_)
    print("Shape predict_proba C:", pipeline_final.predict_proba(X_C).shape)

    t0 = time.time()
    y_pred_C = pipeline_final.predict(X_C)
    score_time_C = time.time() - t0

    metricas_C = calcular_metricas_clasificacion(
        modelo=pipeline_final,
        X_eval=X_C,
        y_eval=y_C,
        y_pred=y_pred_C,
        labels_globales=labels_globales,
        score_time=score_time_C
    )

    fila_holdout = {
        "numero_experimento": 1,
        "descripcion_experimento": DESCRIPCION_EXPERIMENTO,
        "scaler": SCALER_NAME,
        "rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "n": int(target_n),

        "train_original_rows_80": int(df_train.shape[0]),
        "train_balanceado_rows_80": int(df_train_final_balanceado.shape[0]),
        "test_rows_C": int(df_test.shape[0]),

        "C_fit_time": float(fit_time_final),
        "C_score_time": float(score_time_C),
    }

    for nombre, valor in metricas_C.items():
        fila_holdout[f"C_{nombre}"] = valor

    resultados_holdout.append(fila_holdout)

    print(
        f"C FINAL -> acc={metricas_C['accuracy']:.6f}, f1_macro={metricas_C['f1_macro']:.6f}, "
        f"mcc={metricas_C['mcc']:.6f}, fpr_macro={metricas_C['fpr_macro']:.6f}, "
        f"fnr_macro={metricas_C['fnr_macro']:.6f}, auc={metricas_C['roc_auc']:.6f}, "
        f"latency={metricas_C['detection_latency_seconds']:.9f}s"
    )
    print()



####################################################################################################
N = 10000 instancias por clase
####################################################################################################
N=10000 | CV FOLD 1/5


Estrategia de rebalanceo: RUS_SMOTE
Distribución antes del rebalanceo:
LABEL
0     288000
1     288000
2     127271
3      92927
4      92502
5      89456
6      60191
7      26500
8       6341
9       1107
10       355
11       146
12        54
13        35
14        34
Name: count, dtype: int64

Distribución después del undersampling:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8      6341
9      1107
10      355
11      146
12       54
13       35
14       34
Name: count, dtype: int64



SMOTE aplicado con k_neighbors=5
Distribución después de SMOTE:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
14    10000
Name: count, dtype: int64

No se ha aplicado ENN.

Distribución final después del rebalanceo:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
14    10000
Name: count, dtype: int64
Shape final: (150000, 55)



Clases en A: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Clases en B: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Clases del modelo: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Shape predict_proba B: (268230, 15)


B -> acc=0.078817, f1_macro=0.097473, mcc=0.112126, fpr_macro=0.061501, fnr_macro=0.764588, auc=0.699600, latency=0.000000532s

N=10000 | CV FOLD 2/5


...

B -> acc=0.108064, f1_macro=0.093258, mcc=0.137490, fpr_macro=0.060250, fnr_macro=0.740958, auc=0.698205, latency=0.000000535s

N=10000 | MODELO FINAL SOBRE 80% TRAIN + HOLD-OUT 20% TEST
Estrategia de rebalanceo: RUS_SMOTE
Distribución antes del rebalanceo:
LABEL
0     360000
1     360000
2     159089
3     116159
4     115628
5     111820
6      75238
7      33125
8       7926
9       1384
10       444
11       182
12        67
13        44
14        43
Name: count, dtype: int64



Distribución después del undersampling:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8      7926
9      1384
10      444
11      182
12       67
13       44
14       43
Name: count, dtype: int64



SMOTE aplicado con k_neighbors=5
Distribución después de SMOTE:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
14    10000
Name: count, dtype: int64

No se ha aplicado ENN.

Distribución final después del rebalanceo:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
14    10000
Name: count, dtype: int64
Shape final: (150000, 55)



Clases en train final: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Clases en C: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Clases del modelo final: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Shape predict_proba C: (335288, 15)


C FINAL -> acc=0.178637, f1_macro=0.148497, mcc=0.239267, fpr_macro=0.055102, fnr_macro=0.769949, auc=0.707578, latency=0.000000542s



In [12]:
df_folds = pd.DataFrame(resultados_folds)
df_holdout = pd.DataFrame(resultados_holdout)

print("Resultados CV por fold:")
display(df_folds)

print("Resultados hold-out final:")
display(df_holdout)


Resultados CV por fold:


,numero_experimento,descripcion_experimento,scaler,rebalanceo,n,fold,train_original_rows_A,train_balanceado_rows_A,val_rows_B,B_fit_time,...,B_precision_macro,B_recall_macro,B_f1_macro,B_mcc,B_roc_auc,B_detection_latency_seconds,B_fpr_macro,B_fnr_macro,B_fpr_weighted,B_fnr_weighted
0,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,1,1072919,150000,268230,4.564563,...,0.179222,0.235412,0.097473,0.112126,0.699600,5.322682e-07,0.061501,0.764588,0.001328,0.921183
1,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,2,1072919,150000,268230,4.552030,...,0.166496,0.230380,0.094420,0.115080,0.740221,5.258622e-07,0.061023,0.769620,0.001159,0.914193
2,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,3,1072919,150000,268230,4.388460,...,0.162086,0.227318,0.099186,0.119978,0.712379,5.287314e-07,0.060715,0.772682,0.005197,0.905529
3,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,4,1072919,150000,268230,5.523758,...,0.160828,0.232138,0.094510,0.105190,0.688690,5.353347e-07,0.061911,0.767862,0.001851,0.926820
4,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,5,1072920,150000,268229,5.848423,...,0.113166,0.259042,0.093258,0.137490,0.698205,5.348150e-07,0.060250,0.740958,0.011811,0.891936


Resultados hold-out final:


,numero_experimento,descripcion_experimento,scaler,rebalanceo,n,train_original_rows_80,train_balanceado_rows_80,test_rows_C,C_fit_time,C_score_time,...,C_precision_macro,C_recall_macro,C_f1_macro,C_mcc,C_roc_auc,C_detection_latency_seconds,C_fpr_macro,C_fnr_macro,C_fpr_weighted,C_fnr_weighted
0,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,1341149,150000,335288,4.737045,0.181784,...,0.205635,0.230051,0.148497,0.239267,0.707578,5.421732e-07,0.055102,0.769949,0.005174,0.821363


In [13]:
resumenes = []

for target_n, df_n in df_folds.groupby("n"):
    resumen_B = resumir_metricas_cv(df_n, prefijo="B")
    fila_holdout = df_holdout[df_holdout["n"] == target_n].iloc[0]
    resumen_C = extraer_metricas_holdout(fila_holdout)

    fila = {
        "numero_experimento": 1,
        "descripcion_experimento": DESCRIPCION_EXPERIMENTO,
        "scaler": SCALER_NAME,
        "rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "n": int(target_n),
    }

    # B = validación CV: media, desviación típica y varianza
    for k, v in resumen_B.items():
        fila[f"B_{k}"] = v

    # C = hold-out final: valor único, sin media ni desviación típica
    for k, v in resumen_C.items():
        fila[f"C_{k}"] = v

    resumenes.append(fila)

df_resumen = pd.DataFrame(resumenes).sort_values("n").reset_index(drop=True)

df_resumen


,numero_experimento,descripcion_experimento,scaler,rebalanceo,n,B_accuracy_mean,B_accuracy_std,B_accuracy_var,B_precision_weighted_mean,B_precision_weighted_std,...,C_f1_macro,C_fpr_weighted,C_fnr_weighted,C_fpr_macro,C_fnr_macro,C_mcc,C_roc_auc,C_detection_latency_seconds,C_fit_time,C_score_time
0,1,RobustScaler + RUS/SMOTE,RobustScaler,RUS_SMOTE,10000,0.088068,0.013722,0.000188,0.32333,0.049084,...,0.148497,0.005174,0.821363,0.055102,0.769949,0.239267,0.707578,5.421732e-07,4.737045,0.181784


In [14]:
# Tabla en el formato que quieres rellenar.
# IMPORTANTE:
# - En B sí hay μ y σ porque son 5 folds de validación.
# - En C NO hay μ ni σ porque el hold-out se evalúa una sola vez con el modelo final.

def crear_tabla_publicacion_cv(df_resumen, prefijo):
    columnas = {
        f"{prefijo}_accuracy_mean": "μ ACCURACY",
        f"{prefijo}_accuracy_std": "σ ACCURACY",

        f"{prefijo}_precision_weighted_mean": "μ PRECISION WEIGHTED",
        f"{prefijo}_precision_weighted_std": "σ PRECISION WEIGHTED",

        f"{prefijo}_recall_weighted_mean": "μ RECALL WEIGHTED",
        f"{prefijo}_recall_weighted_std": "σ RECALL WEIGHTED",

        f"{prefijo}_f1_weighted_mean": "μ F1 WEIGHTED",
        f"{prefijo}_f1_weighted_std": "σ F1 WEIGHTED",

        f"{prefijo}_precision_macro_mean": "μ PRECISION MACRO",
        f"{prefijo}_precision_macro_std": "σ PRECISION MACRO",

        f"{prefijo}_recall_macro_mean": "μ RECALL MACRO",
        f"{prefijo}_recall_macro_std": "σ RECALL MACRO",

        f"{prefijo}_f1_macro_mean": "μ F1 MACRO",
        f"{prefijo}_f1_macro_std": "σ F1 MACRO",

        f"{prefijo}_fpr_weighted_mean": "μ FPR WEIGHTED",
        f"{prefijo}_fpr_weighted_std": "σ FPR WEIGHTED",

        f"{prefijo}_fnr_weighted_mean": "μ FNR WEIGHTED",
        f"{prefijo}_fnr_weighted_std": "σ FNR WEIGHTED",

        f"{prefijo}_fpr_macro_mean": "μ FPR MACRO",
        f"{prefijo}_fpr_macro_std": "σ FPR MACRO",

        f"{prefijo}_fnr_macro_mean": "μ FNR MACRO",
        f"{prefijo}_fnr_macro_std": "σ FNR MACRO",

        f"{prefijo}_mcc_mean": "μ MCC",
        f"{prefijo}_mcc_std": "σ MCC",

        f"{prefijo}_roc_auc_mean": "μ AUC-ROC",
        f"{prefijo}_roc_auc_std": "σ AUC-ROC",

        f"{prefijo}_detection_latency_seconds_mean": "μ DETECTION LATENCY (S)",
        f"{prefijo}_detection_latency_seconds_std": "σ DETECTION LATENCY (S)",

        f"{prefijo}_fit_time_mean": "μ FIT TIME",
        f"{prefijo}_fit_time_var": "σ² FIT TIME",

        f"{prefijo}_score_time_mean": "μ SCORE TIME",
        f"{prefijo}_score_time_var": "σ² SCORE TIME",
    }

    cols_existentes = [c for c in columnas.keys() if c in df_resumen.columns]
    df_tabla = df_resumen[cols_existentes].rename(columns=columnas)

    cols_num = df_tabla.select_dtypes(include=[np.number]).columns
    df_tabla[cols_num] = df_tabla[cols_num].round(6)

    return df_tabla


def crear_tabla_publicacion_holdout(df_resumen):
    columnas = {
        "C_accuracy": "ACCURACY",
        "C_precision_weighted": "PRECISION WEIGHTED",
        "C_recall_weighted": "RECALL WEIGHTED",
        "C_f1_weighted": "F1 WEIGHTED",

        "C_precision_macro": "PRECISION MACRO",
        "C_recall_macro": "RECALL MACRO",
        "C_f1_macro": "F1 MACRO",

        "C_fpr_weighted": "FPR WEIGHTED",
        "C_fnr_weighted": "FNR WEIGHTED",
        "C_fpr_macro": "FPR MACRO",
        "C_fnr_macro": "FNR MACRO",

        "C_mcc": "MCC",
        "C_roc_auc": "AUC-ROC",
        "C_detection_latency_seconds": "DETECTION LATENCY (S)",

        "C_fit_time": "FIT TIME",
        "C_score_time": "SCORE TIME",
    }

    cols_existentes = [c for c in columnas.keys() if c in df_resumen.columns]
    df_tabla = df_resumen[cols_existentes].rename(columns=columnas)

    cols_num = df_tabla.select_dtypes(include=[np.number]).columns
    df_tabla[cols_num] = df_tabla[cols_num].round(6)

    return df_tabla


def imprimir_tabla_copiable(df_tabla):
    print("\t".join(df_tabla.columns))

    for _, fila in df_tabla.iterrows():
        valores = []

        for valor in fila:
            if pd.isna(valor):
                valores.append("")
            elif isinstance(valor, (int, np.integer)):
                valores.append(str(valor))
            elif isinstance(valor, (float, np.floating)):
                valores.append(f"{valor:.6f}")
            else:
                valores.append(str(valor))

        print("\t".join(valores))


df_tabla_B = crear_tabla_publicacion_cv(df_resumen, prefijo="B")
df_tabla_C = crear_tabla_publicacion_holdout(df_resumen)

print("TABLA B - VALIDACIÓN CV")
imprimir_tabla_copiable(df_tabla_B)

print()
print("TABLA C - HOLD-OUT/TEST FINAL")
imprimir_tabla_copiable(df_tabla_C)


TABLA B - VALIDACIÓN CV
μ ACCURACY	σ ACCURACY	μ PRECISION WEIGHTED	σ PRECISION WEIGHTED	μ RECALL WEIGHTED	σ RECALL WEIGHTED	μ F1 WEIGHTED	σ F1 WEIGHTED	μ PRECISION MACRO	σ PRECISION MACRO	μ RECALL MACRO	σ RECALL MACRO	μ F1 MACRO	σ F1 MACRO	μ FPR WEIGHTED	σ FPR WEIGHTED	μ FNR WEIGHTED	σ FNR WEIGHTED	μ FPR MACRO	σ FPR MACRO	μ FNR MACRO	σ FNR MACRO	μ MCC	σ MCC	μ AUC-ROC	σ AUC-ROC	μ DETECTION LATENCY (S)	σ DETECTION LATENCY (S)	μ FIT TIME	σ² FIT TIME	μ SCORE TIME	σ² SCORE TIME
0.088068	0.013722	0.323330	0.049084	0.088068	0.013722	0.115163	0.017316	0.156359	0.025218	0.236858	0.012743	0.095769	0.002464	0.004269	0.004525	0.911932	0.013722	0.061080	0.000651	0.763142	0.012743	0.117973	0.012154	0.707819	0.019979	0.000001	0.000000	4.975447	0.438848	0.142538	0.000001

TABLA C - HOLD-OUT/TEST FINAL
ACCURACY	PRECISION WEIGHTED	RECALL WEIGHTED	F1 WEIGHTED	PRECISION MACRO	RECALL MACRO	F1 MACRO	FPR WEIGHTED	FNR WEIGHTED	FPR MACRO	FNR MACRO	MCC	AUC-ROC	DETECTION LATENCY (S)	FIT TIME	SCORE TIME
0.178637	

In [15]:
ruta_folds_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_FOLDS_CSV
ruta_holdout_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_HOLDOUT_CSV
ruta_resumen_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_RESUMEN_CSV

df_folds.to_csv(ruta_folds_csv, index=False)
df_holdout.to_csv(ruta_holdout_csv, index=False)
df_resumen.to_csv(ruta_resumen_csv, index=False)

df_tabla_B.to_csv(RUTA_RESULTADOS / f"{NOMBRE_EXPERIMENTO}__tabla_B_validacion_cv.csv", index=False)
df_tabla_C.to_csv(RUTA_RESULTADOS / f"{NOMBRE_EXPERIMENTO}__tabla_C_holdout_final.csv", index=False)

print("Resultados CV por fold guardados en:")
print(ruta_folds_csv.resolve())
print()

print("Resultados hold-out final guardados en:")
print(ruta_holdout_csv.resolve())
print()

print("Resumen B y C guardado en:")
print(ruta_resumen_csv.resolve())
print()

print("Tablas para copiar guardadas en:")
print((RUTA_RESULTADOS / f"{NOMBRE_EXPERIMENTO}__tabla_B_validacion_cv.csv").resolve())
print((RUTA_RESULTADOS / f"{NOMBRE_EXPERIMENTO}__tabla_C_holdout_final.csv").resolve())


Resultados CV por fold guardados en:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1__folds_B.csv

Resultados hold-out final guardados en:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1__holdout_C.csv

Resumen B y C guardado en:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1__tabla_resumen_B_y_C.csv

Tablas para copiar guardadas en:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1__tabla_B_validacion_cv.csv
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_Robu

In [16]:
summary = {
    "experimento": NOMBRE_EXPERIMENTO,
    "descripcion_experimento": DESCRIPCION_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "pipeline": {
        "descripcion": "CV solo sobre el 80% de train. Hold-out final fuera del CV, entrenando un modelo final con todo el 80% y evaluando una sola vez sobre el 20% de test.",
        "cv": "5 folds sobre train: 64% entrenamiento A y 16% validacion B por fold",
        "holdout": "Modelo final entrenado con el 80% completo y evaluado una unica vez sobre test C"
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_values": N_VALUES,
        "scaler": SCALER_NAME,
        "rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "modelo": MODEL_DISPLAY_NAME,
        "parametros_modelo": MODEL_PARAMS,
        "n_components_pca": N_COMPONENTS_PCA,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "resultados_cv_por_fold": df_folds.to_dict(orient="records"),
    "resultados_holdout_final": df_holdout.to_dict(orient="records"),
    "resumen_por_n": df_resumen.to_dict(orient="records")
}

ruta_summary_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_RESUMEN_JSON

with open(ruta_summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("Resumen JSON guardado en:")
print(ruta_summary_json.resolve())


Resumen JSON guardado en:
/home/javier/TFG_FASE_1/04_experimentos/logs/resultados/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1/CIC18__split__v1__fase1_RobustScaler_RUS_SMOTE_pca_mlp__v1__summary_B_y_C.json


In [17]:
imprimir_tabla_copiable(df_tabla_B)

μ ACCURACY	σ ACCURACY	μ PRECISION WEIGHTED	σ PRECISION WEIGHTED	μ RECALL WEIGHTED	σ RECALL WEIGHTED	μ F1 WEIGHTED	σ F1 WEIGHTED	μ PRECISION MACRO	σ PRECISION MACRO	μ RECALL MACRO	σ RECALL MACRO	μ F1 MACRO	σ F1 MACRO	μ FPR WEIGHTED	σ FPR WEIGHTED	μ FNR WEIGHTED	σ FNR WEIGHTED	μ FPR MACRO	σ FPR MACRO	μ FNR MACRO	σ FNR MACRO	μ MCC	σ MCC	μ AUC-ROC	σ AUC-ROC	μ DETECTION LATENCY (S)	σ DETECTION LATENCY (S)	μ FIT TIME	σ² FIT TIME	μ SCORE TIME	σ² SCORE TIME
0.088068	0.013722	0.323330	0.049084	0.088068	0.013722	0.115163	0.017316	0.156359	0.025218	0.236858	0.012743	0.095769	0.002464	0.004269	0.004525	0.911932	0.013722	0.061080	0.000651	0.763142	0.012743	0.117973	0.012154	0.707819	0.019979	0.000001	0.000000	4.975447	0.438848	0.142538	0.000001


In [18]:
imprimir_tabla_copiable(df_tabla_C)

ACCURACY	PRECISION WEIGHTED	RECALL WEIGHTED	F1 WEIGHTED	PRECISION MACRO	RECALL MACRO	F1 MACRO	FPR WEIGHTED	FNR WEIGHTED	FPR MACRO	FNR MACRO	MCC	AUC-ROC	DETECTION LATENCY (S)	FIT TIME	SCORE TIME
0.178637	0.408323	0.178637	0.207259	0.205635	0.230051	0.148497	0.005174	0.821363	0.055102	0.769949	0.239267	0.707578	0.000001	4.737045	0.181784
